Transformer Encoder-Decoder
====


  - encoder self-attention: bidirectional, no causal mask
  - decoder self-attention: causal mask
  - decoder cross-attention: queries from decoder, keys/values from encoder
  - feed-forward network after attention
  - stacked encoder and decoder blocks

https://github.com/huggingface/transformers/blob/main/src/transformers/models/bart/modeling_bart.py

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from tokenizers import Tokenizer

In [2]:
embedding_dim = 512
context_length = 1024
num_heads = 8
dropout_rate = 0.1

sqrt_d_model = embedding_dim ** 0.5

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_dir = repo_root / "data" / "tokenizers"
tokenizer_dir.mkdir(parents=True, exist_ok=True)

tokenizer_path = tokenizer_dir / "tatoeba_bpe_en_de.json"
print(f"Tokenizer path: {tokenizer_path}")

Tokenizer path: /home/pyml/git_kmehr/llm-recipes/data/tokenizers/tatoeba_bpe_en_de.json


In [3]:
# Load the tokenizer from disk, like we would in later notebooks.
tokenizer = Tokenizer.from_file(str(tokenizer_path))

n_vocab = tokenizer.get_vocab_size()
print(f"Vocabulary size: {n_vocab:,}")

pad_token_id = tokenizer.token_to_id("[PAD]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")
unk_token_id = tokenizer.token_to_id("[UNK]")
print(f"PAD token id: {pad_token_id}")
print(f"BOS token id: {bos_token_id}")
print(f"EOS token id: {eos_token_id}")
print(f"UNK token id: {unk_token_id}")


# running an example
text_en = "The encoder uses attention."
text_de = "Der Encoder verwendet Attention."

encoded_en = tokenizer.encode(text_en)
encoded_de = tokenizer.encode(text_de)

print("English tokens:", encoded_en.tokens)
print("English ids:", encoded_en.ids)
print("German tokens:", encoded_de.tokens)
print("German ids:", encoded_de.ids)

Vocabulary size: 16,000
PAD token id: 0
BOS token id: 1
EOS token id: 2
UNK token id: 3
English tokens: ['[BOS]', 'The', 'en', 'co', 'der', 'uses', 'attention', '.', '[EOS]']
English ids: [1, 220, 158, 331, 197, 5893, 4537, 16, 2]
German tokens: ['[BOS]', 'Der', 'En', 'co', 'der', 'verwendet', 'Atten', 'tion', '.', '[EOS]']
German ids: [1, 328, 375, 331, 197, 7735, 14963, 2596, 16, 2]


### 1. Prepare the inputs


In [4]:
tok_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-torch.log(torch.tensor(10000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


pos_embeddings = SinusoidalPositionEmbedding(
    context_length=context_length, embedding_dim=embedding_dim
)


#### Source sequences

In [5]:
source_texts = [
    "Transformers use attention to process the input sequence.",
    "The encoder uses attention to create a representation of the input sequence.",
]

source_encoded_batch = tokenizer.encode_batch(source_texts)
source_token_ids = [encoded.ids for encoded in source_encoded_batch]
print(f"source_token_ids lengths: {[len(ids) for ids in source_token_ids]}")

# generate padding mask
pad_token_id = tokenizer.token_to_id("[PAD]")
print(f"pad_token_id ([PAD]): {pad_token_id}")

source_max_len = max(len(ids) for ids in source_token_ids)
padded_source_ids = [
    ids + [pad_token_id] * (source_max_len - len(ids))
    for ids in source_token_ids
]
print(f"padded_source_ids: {padded_source_ids}")

source_ids_tensor = torch.tensor(padded_source_ids, dtype=torch.long)

source_padding_mask = source_ids_tensor.eq(pad_token_id)

print("source_ids_tensor:", source_ids_tensor.shape)      # [batch_size, max_len]
print("source_padding_mask:", source_padding_mask.shape)  # [batch_size, max_len]

print("source_ids_tensor:\n", source_ids_tensor)
print("source_padding_mask:\n", source_padding_mask)


source_token_ids lengths: [15, 20]
pad_token_id ([PAD]): 0
padded_source_ids: [[1, 14393, 12809, 80, 554, 4537, 181, 10476, 180, 160, 959, 11841, 1785, 16, 2, 0, 0, 0, 0, 0], [1, 220, 158, 331, 197, 5893, 4537, 181, 14888, 62, 12763, 400, 201, 180, 160, 959, 11841, 1785, 16, 2]]
source_ids_tensor: torch.Size([2, 20])
source_padding_mask: torch.Size([2, 20])
source_ids_tensor:
 tensor([[    1, 14393, 12809,    80,   554,  4537,   181, 10476,   180,   160,
           959, 11841,  1785,    16,     2,     0,     0,     0,     0,     0],
        [    1,   220,   158,   331,   197,  5893,  4537,   181, 14888,    62,
         12763,   400,   201,   180,   160,   959, 11841,  1785,    16,     2]])
source_padding_mask:
 tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, Fals

In [6]:
source_tok_emb = tok_embeddings(source_ids_tensor)

batch_size, source_len = source_ids_tensor.shape

source_positions = torch.arange(source_len, dtype=torch.long, device=source_ids_tensor.device)
source_pos_emb = pos_embeddings(source_positions)  # [src_len, embedding_dim]
print(f"pos_emb.shape: {source_pos_emb.shape}")    # [src_len, 512]

# scale and add positional embeddings to token embeddings
encoder_input = source_tok_emb * sqrt_d_model + source_pos_emb
# The original Transformer does this because token embeddings and sinusoidal position embeddings are added directly.

print(f"Encoder input: {encoder_input.shape}")

# note: we should also apply embedding dropout here, but we will skip it for simplicity.
# embedding_dropout = nn.Dropout(p=dropout_rate)
# encoder_input = embedding_dropout(source_tok_emb * sqrt_d_model + source_pos_emb)

pos_emb.shape: torch.Size([20, 512])
Encoder input: torch.Size([2, 20, 512])


#### Target sequences

**Teacher forcing**
- target ids
- decoder input ids
- labels

In [7]:
target_texts = [
    "Transformer verwenden Attention, um die Eingabesequenz zu verarbeiten.",
    "Der Encoder verwendet Attention, um eine Repräsentation der Eingabesequenz zu erzeugen.",
]

target_encoded_batch = tokenizer.encode_batch(target_texts)
target_token_ids = [encoded.ids for encoded in target_encoded_batch]
print(f"target_ids lengths: {[len(ids) for ids in target_token_ids]}")

target_max_len = max(len(ids) for ids in target_token_ids)
padded_target_ids = [
    ids + [pad_token_id] * (target_max_len - len(ids))
    for ids in target_token_ids
]

# Complete padded target sequences:
# [BOS, token_1, token_2, ..., EOS, PAD, PAD]
target_ids_tensor = torch.tensor(padded_target_ids, dtype=torch.long)
# Remove the final position for decoder inputs.
decoder_input_ids = target_ids_tensor[:, :-1]
# Remove the initial BOS position for expected outputs.
labels = target_ids_tensor[:, 1:]

print(f"target_ids_tensor: {target_ids_tensor.shape}")  # [batch_size, target_len]
print(f"decoder_input_ids: {decoder_input_ids.shape}")  # [batch_size, target_len - 1]
print(f"labels: {labels.shape}")                        # [batch_size, target_len - 1]

# generate padding mask
target_padding_mask = target_ids_tensor.eq(pad_token_id)
decoder_padding_mask = target_padding_mask[:, :-1]

print("decoder_padding_mask:", decoder_padding_mask.shape)  # [batch_size, target_len - 1]

print("decoder_input_ids:\n", decoder_input_ids)
print("labels:\n", labels)
print("decoder_padding_mask:\n", decoder_padding_mask)

# target embeddings
decoder_tok_emb = tok_embeddings(decoder_input_ids)

batch_size, decoder_len = decoder_input_ids.shape

decoder_positions = torch.arange(decoder_len, dtype=torch.long, device=decoder_input_ids.device)
decoder_pos_emb = pos_embeddings(decoder_positions)       # [decoder_len, embedding_dim]
print(f"decoder_pos_emb.shape: {decoder_pos_emb.shape}")  # [decoder_len, 512]

# scale and add positional embeddings to token embeddings
decoder_input = decoder_tok_emb * sqrt_d_model + decoder_pos_emb

print(f"Decoder input: {decoder_input.shape}")

target_ids lengths: [20, 27]
target_ids_tensor: torch.Size([2, 27])
decoder_input_ids: torch.Size([2, 26])
labels: torch.Size([2, 26])
decoder_padding_mask: torch.Size([2, 26])
decoder_input_ids:
 tensor([[    1, 14393, 12809,  7580, 14963,  2596,    14,   253,   208, 10681,
           241,   605,   441,   158,    87,   196,   203,  2266,    16,     2,
             0,     0,     0,     0,     0,     0],
        [    1,   328,   375,   331,   197,  7735, 14963,  2596,    14,   253,
           210,   469,  6784,  1425,   400,   197, 10681,   241,   605,   441,
           158,    87,   196,   157,  6572,    16]])
labels:
 tensor([[14393, 12809,  7580, 14963,  2596,    14,   253,   208, 10681,   241,
           605,   441,   158,    87,   196,   203,  2266,    16,     2,     0,
             0,     0,     0,     0,     0,     0],
        [  328,   375,   331,   197,  7735, 14963,  2596,    14,   253,   210,
           469,  6784,  1425,   400,   197, 10681,   241,   605,   441,   158,
     

### 2. Multi-head Attention (MHA)


The two masks serve different purposes:

- causal mask  
    Prevents cheating by hiding future decoder tokens.

- decoder padding mask    
    Prevents attention to PAD tokens in the decoder input.  

- encoder/source padding mask  
    Prevents cross-attention to PAD tokens in the source sequence.

In [8]:
# combining three types of MHA into one class: standard, cross-attention, and causal attention
class MultiheadAttention(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            is_crossattention=False,
            is_causal=False,
            dropout_rate=0.1
        ):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        assert not (
            is_crossattention and is_causal
        ), "Cross-attention should not use a causal mask"
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // num_heads
        self.is_crossattention = is_crossattention
        self.is_causal = is_causal

        # if self.is_crossattention:
        #     self.w_attn = nn.Linear(embedding_dim, 2 * embedding_dim)  # for k & v
        #     self.w_q = nn.Linear(embedding_dim, embedding_dim)
        # else:
        #     self.w_attn = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)

        self.attn_dropout = nn.Dropout(dropout_rate)

        if self.is_causal:
            mask = torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            )
            self.register_buffer("causal_mask", mask)

    def forward(self, hidden_states, key_value_states=None, padding_mask=None):
        batch_size, query_len, _ = hidden_states.size()

        if self.is_crossattention:
            assert key_value_states is not None, "key_value_states must be provided for cross-attention"
        else:
            assert key_value_states is None, "key_value_states are only used for cross-attention"
            key_value_states = hidden_states

        assert key_value_states.size(0) == batch_size, "query and key/value batch sizes must match"
        assert key_value_states.size(2) == self.embedding_dim, "key/value embedding dimension must match"
        key_value_len = key_value_states.size(1)

        # Linear projections
        # if self.is_crossattention:
        #     q = self.w_q(x)
        #     k, v = self.w_attn(kv_states).split(self.embedding_dim, dim=-1)
        # else:
        #     q, k, v = self.w_attn(x).split(self.embedding_dim, dim=-1)
        q = self.w_q(hidden_states)
        k = self.w_k(key_value_states)
        v = self.w_v(key_value_states)
        # current shapes: [batch_size, seq_len, embedding_dim]

        # split the embedding dimension into multiple heads
        # -> shape: [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Causal mask
        if self.is_causal:
            assert query_len <= self.causal_mask.size(0), "sequence length exceeds context_length"
            assert key_value_len <= self.causal_mask.size(0), "sequence length exceeds context_length"
            causal_mask = self.causal_mask[:query_len, :key_value_len]
            attn_scores = attn_scores.masked_fill(
                causal_mask.unsqueeze(0).unsqueeze(0),
                float('-inf')
            )

        # attention mask (for padding tokens)
        if padding_mask is not None:
            assert padding_mask.shape == (batch_size, key_value_len)
            assert padding_mask.dtype == torch.bool
            # padding_mask shape: [batch_size, key_value_len]
            attn_scores = attn_scores.masked_fill(
                padding_mask.unsqueeze(1).unsqueeze(2),
                float("-inf"),
            )

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = torch.matmul(attn_weights, v)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, query_len, -1)
        output = self.w_o(output)

        return output
    
# testing:
mha_layer = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    dropout_rate=0.1
)
output_mha = mha_layer(encoder_input, padding_mask=source_padding_mask)
print("MHA (encoder, non-causal):")
print(f" - Input shape → {encoder_input.shape}")
print(f" - MHA Output shape → {output_mha.shape}")

mha_layer_causal = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=False,
    is_causal=True,
    dropout_rate=0.1
)

output_mha_causal = mha_layer_causal(decoder_input, padding_mask=decoder_padding_mask)
print("\nMHA (causal):")
print(f"- Input shape → {decoder_input.shape}")
print(f" - MHA output shape → {output_mha_causal.shape}")

mha_layer_cross = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=True,
    is_causal=False,
    dropout_rate=0.1
)
output_mha_cross = mha_layer_cross(
    decoder_input,
    key_value_states=encoder_input,
    padding_mask=source_padding_mask
)
print("\nMHA (cross):")
print(f"- Input shape → {decoder_input.shape}")
print(f"- KV-states shape → {encoder_input.shape}")
print(f"- MHA output shape → {output_mha_cross.shape}")


MHA (encoder, non-causal):
 - Input shape → torch.Size([2, 20, 512])
 - MHA Output shape → torch.Size([2, 20, 512])

MHA (causal):
- Input shape → torch.Size([2, 26, 512])
 - MHA output shape → torch.Size([2, 26, 512])

MHA (cross):
- Input shape → torch.Size([2, 26, 512])
- KV-states shape → torch.Size([2, 20, 512])
- MHA output shape → torch.Size([2, 26, 512])


### 3. Transformer Encoder Block

In [9]:
# MHA -> dropout -> residual -> layer norm -> FFN -> dropout -> residual -> layer norm

class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.mha = MultiheadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            is_crossattention=False,
            is_causal=False,
            dropout_rate=dropout_rate
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),  # original transformer uses ReLU
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.resid_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x, padding_mask=None):
        # Multi-head attention
        attn_output = self.mha(x, padding_mask=padding_mask)
        x = self.layer_norm1(x + self.resid_dropout(attn_output))  # residual connection + layer norm
        # Feed-forward network
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + self.ffn_dropout(ffn_output))  # residual connection + layer norm
        return x
    

# testing:
encoder_block = EncoderBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    context_length=context_length,
    dropout_rate=0.1
)

output_encoder_block = encoder_block(encoder_input, padding_mask=source_padding_mask)
print(f"Encoder block output shape: {output_encoder_block.shape}")

Encoder block output shape: torch.Size([2, 20, 512])


### 4. Transformer Decoder Block

In [10]:
class DecoderBlock(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            dropout_rate=0.1
    ):
        super().__init__()
        self.causal_mha = MultiheadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            is_crossattention=False,
            is_causal=True,
            dropout_rate=dropout_rate
        )
        self.cross_mha = MultiheadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            is_crossattention=True,
            is_causal=False,
            dropout_rate=dropout_rate
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.resid_dropout1 = nn.Dropout(dropout_rate)
        self.resid_dropout2 = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.layer_norm3 = nn.LayerNorm(embedding_dim)

    def forward(
            self,
            x,
            encoder_hidden_states,
            decoder_padding_mask=None,
            encoder_padding_mask=None
        ):
        # Self-attention
        attn_output = self.causal_mha(x, padding_mask=decoder_padding_mask)
        x = self.layer_norm1(x + self.resid_dropout1(attn_output))  # residual connection + layer norm

        # Cross-attention
        attn_output = self.cross_mha(
            x,
            key_value_states=encoder_hidden_states,
            padding_mask=encoder_padding_mask
        )
        x = self.layer_norm2(x + self.resid_dropout2(attn_output))

        # FFN
        ffn_output = self.ffn(x)
        x = self.layer_norm3(x + self.ffn_dropout(ffn_output))

        return x


In [11]:
# testing
decoder_block = DecoderBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    context_length=context_length,
    dropout_rate=dropout_rate
)

output_decoder_block = decoder_block(
    decoder_input,
    encoder_hidden_states=output_encoder_block,
    decoder_padding_mask=decoder_padding_mask,
    encoder_padding_mask=source_padding_mask
)
print(f"Decoder block output shape: {output_decoder_block.shape}")

Decoder block output shape: torch.Size([2, 26, 512])


## Summary and Next Steps

This notebook built the core components of a Transformer encoder-decoder and traced how sequence lengths flow through them.

- A shared token embedding layer maps both source tokens and decoder-input tokens into the model dimension. Sinusoidal positional encodings are added after scaling the token embeddings by $\sqrt{d_{\text{model}}}$.
- Teacher forcing shifts each complete target sequence into `decoder_input_ids = target_ids[:, :-1]` and `labels = target_ids[:, 1:]`. The decoder therefore learns to predict the next token rather than copy the current token.
- Encoder self-attention is bidirectional. Decoder self-attention combines a causal mask with the decoder padding mask. Cross-attention uses decoder states as queries and encoder states as keys and values.
- If the decoder query length is $T$ and the encoder key/value length is $S$, cross-attention scores have shape $[B, H, T, S]$. The source padding mask has shape $[B, S]$ and broadcasts to $[B, 1, 1, S]$.
- Attention output preserves the query length. Consequently, cross-attention with decoder queries of length $T$ returns $[B, T, d_{\text{model}}]$, even when the encoder length $S$ is different.
- One encoder block applies non-causal self-attention followed by a feed-forward network. One decoder block applies causal self-attention, encoder-decoder cross-attention, and a feed-forward network. Each sublayer uses dropout, a residual connection, and layer normalization.

In the following notebook, we will build on these components by stacking multiple encoder and decoder blocks, tying the output projection to the shared token embedding matrix, and computing the padding-aware next-token loss.